<h1 align='center'> Spam Clasifier </h1>

In [1]:
!pip install matplotlib plotly --upgrade --quiet
!pip install seaborn --upgrade --quiet
!pip install scikit-learn --upgrade --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 87.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 99.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 86.4 MB/s eta 0:00:00:00:010:01


In [2]:
!pip install opendatasets --upgrade --quiet

In [3]:
import numpy as np
import pandas as pd
import opendatasets as op
import matplotlib.pyplot as plt
import seaborn as sns
import os 

In [24]:
op.download("https://www.kaggle.com/datasets/ashfakyeafi/spam-email-classification")

Skipping, found downloaded files in "./spam-email-classification" (use force=True to force download)


In [25]:
os.listdir('./spam-email-classification')

['email.csv']

In [26]:
df=pd.read_csv('./spam-email-classification/email.csv')

In [27]:
df['Category'].value_counts()

,count
Category,
ham,4825
spam,747
"{""mode"":""full""",1


15% spam from entire data set

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5573 entries, 0 to 5572
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5573 non-null   object
 1   Message   5573 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [29]:
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [30]:
print("Columns:", df.columns.tolist())

Columns: ['Category', 'Message']


In [31]:
print("First Row:\n", df.iloc[-1])

First Row:
 Category     {"mode":"full"
Message     isActive:false}
Name: 5572, dtype: object


In [32]:
df = df[df['Category'].isin(['ham', 'spam'])].copy()

In [33]:
df['label'] = df['Category'].map({'ham': 0, 'spam': 1})

In [34]:
df

,Category,Message,label
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will ü b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


In [35]:
import nltk
from nltk.corpus import stopwords


nltk.download('stopwords')
stop_word=set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [36]:
def clean_text(text):
    import re
    text=text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words=text.split()
    clean_words= [word for word in words if word not in stop_word]
    return " ".join(clean_words)

In [37]:
df['cleaned_msg']=df['Message'].apply(clean_text)

In [38]:
df

,Category,Message,label,cleaned_msg
0,ham,"Go until jurong point, crazy.. Available only ...",0,go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,0,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,free entry wkly comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,0,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,nah dont think goes usf lives around though
...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1,nd time tried contact u u pound prize claim ea...
5568,ham,Will ü b going to esplanade fr home?,0,b going esplanade fr home
5569,ham,"Pity, * was in mood for that. So...any other s...",0,pity mood soany suggestions
5570,ham,The guy did some bitching but I acted like i'd...,0,guy bitching acted like id interested buying s...


In [40]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(df['cleaned_msg'], df['label'], test_size=0.2, random_state=42,  stratify=df['label'])

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=3000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

Evaluating diffrent Models

In [51]:
def evaluate_models(model, X_tr,Y_tr,X_tes,Y_tes):
    from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
    md=model
    md.fit(X_tr, Y_tr)
    tes_pred=md.predict(X_tes)
    print("Precision:", precision_score(Y_tes, tes_pred))
    print("Recall:   ", recall_score(Y_tes, tes_pred))
    print("Accurassy", accuracy_score(Y_test,tes_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(Y_tes, tes_pred))


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier


In [52]:
evaluate_models(LogisticRegression(), X_train_tfidf,Y_train,X_test_tfidf,Y_test)

Precision: 1.0
Recall:    0.7651006711409396
Accurassy 0.968609865470852

Confusion Matrix:
 [[966   0]
 [ 35 114]]


In [53]:
evaluate_models(RidgeClassifier(), X_train_tfidf,Y_train,X_test_tfidf,Y_test)

Precision: 0.9923076923076923
Recall:    0.8657718120805369
Accurassy 0.9811659192825112

Confusion Matrix:
 [[965   1]
 [ 20 129]]


In [54]:
evaluate_models(RandomForestClassifier(), X_train_tfidf,Y_train,X_test_tfidf,Y_test)

Precision: 1.0
Recall:    0.8322147651006712
Accurassy 0.9775784753363229

Confusion Matrix:
 [[966   0]
 [ 25 124]]


In [55]:
evaluate_models(MultinomialNB(), X_train_tfidf,Y_train,X_test_tfidf,Y_test)

Precision: 0.9915254237288136
Recall:    0.785234899328859
Accurassy 0.9704035874439462

Confusion Matrix:
 [[965   1]
 [ 32 117]]


Test real world emails

In [ ]:
def predict_email(custom_email, model, vectorizer):
    
    cleaned = clean_text(custom_email)
    
    
    vectorized = vectorizer.transform([cleaned])
    
    
    prediction = model.predict(vectorized)[0]
    
    return "SPAM" if prediction == 1 else "HAM (Legitimate)"


Message 1: HAM (Legitimate)
Message 2: SPAM


In [57]:
best_model = RandomForestClassifier(random_state=42)
best_model.fit(X_train_tfidf, Y_train)

RandomForestClassifier(random_state=42)

In [59]:
import joblib


joblib.dump(best_model, 'spam_model.pkl')


joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']